This is a notebook to run the landcover validation and most of the prerequisite steps to get to this point.
The steps included are:
- downloading landcover dataset zip
- extracting data from the landcover dataset
- tiling RGB and remapping/slicing GT masks for verification
- running inference on tiles (runmodel)
- scoring predictions (F1 / IoU CSV)
- generating triplet comparison PNGs

The steps excluded are the model training and related sanity checks. Because of this, it is required for you to have the unet checkpoint file downloaded already.

All paths in markdown steps are relative to the root of the project (e.g. this notebook is in `./landcover_verification/`). This may not be the case for the Python code boxes.

step 0: download the unet checkpoint file and place it in `./checkpoints/pipeline_best_unet_best.pt`

In [13]:
import os
import hashlib
# assert the file exists
assert os.path.exists("../checkpoints/pipeline_best_unet_best.pt")
# assert file integrity
assert hashlib.md5(open("../checkpoints/pipeline_best_unet_best.pt", "rb").read()).hexdigest() == "d6d073007380e0240b88ca91e4e07895"
print("File integrity verified")

File integrity verified


step 1: download zip of dataset v1 from https://landcover.ai.linuxpolska.com/, extract to `./landcover_verification/datasets/landcover_dataset`

In [14]:
# Download LandCover.ai v1 zip into datasets/, extract to landcover_dataset/
from __future__ import annotations

import shutil
import tempfile
import urllib.request
import zipfile
from pathlib import Path

LANDCOVER_V1_URL = "https://landcover.ai.linuxpolska.com/download/landcover.ai.v1.zip"
ZIP_FILENAME = "landcover.ai.v1.zip"

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "landcover_verification" else _cwd
DATASETS_DIR = PROJECT_ROOT / "landcover_verification" / "datasets"
TARGET_DIR = DATASETS_DIR / "landcover_dataset"
ZIP_PATH = DATASETS_DIR / ZIP_FILENAME


def _find_images_masks_root(root: Path) -> Path | None:
    if (root / "images").is_dir() and (root / "masks").is_dir():
        return root
    for child in sorted(root.iterdir()):
        if child.is_dir():
            found = _find_images_masks_root(child)
            if found is not None:
                return found
    return None


DATASETS_DIR.mkdir(parents=True, exist_ok=True)

if (TARGET_DIR / "masks").is_dir() and any((TARGET_DIR / "masks").iterdir()):
    print(f"Skip download: {TARGET_DIR} already has masks.")
else:
    print(f"Downloading {LANDCOVER_V1_URL} …")
    urllib.request.urlretrieve(LANDCOVER_V1_URL, ZIP_PATH)
    print(f"Saved {ZIP_PATH}")

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(tmp_path)
        content_root = _find_images_masks_root(tmp_path)
        if content_root is None:
            raise RuntimeError(
                "Could not find a directory with both 'images/' and 'masks/' inside the zip."
            )
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)
        shutil.move(str(content_root), str(TARGET_DIR))

    print(f"Extracted dataset to {TARGET_DIR}")

Skip download: /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/landcover_dataset already has masks.


step 2: tile RGB images and remap/slice GT masks into `./landcover_verification/datasets/tiled_images` and `./landcover_verification/datasets/remapped_landcover_masks` (tile size 500, same as `run_landcover_verification.sh`). Re-running the next code cell clears those two directories of old GeoTIFFs first, same as the shell default.

In [15]:
# Pipeline paths + step 2: tile and remap (matches run_landcover_verification.sh step 1/4)
from __future__ import annotations

import re
import subprocess
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "landcover_verification" else _cwd
DATASETS_DIR = PROJECT_ROOT / "landcover_verification" / "datasets"

MODEL = "unet"
model_safe = re.sub(r"[^0-9A-Za-z._-]", "_", MODEL)
TILE_SIZE = 500
RUNMODEL_LIMIT = 0
TRIPLET_LIMIT = 3
INFERENCE_PATCH_SIZE = 0
INFERENCE_OVERLAP = 0

RAW_RGB_DIR = DATASETS_DIR / "landcover_dataset" / "images"
RAW_GT_DIR = DATASETS_DIR / "landcover_dataset" / "masks"
TILED_RGB_DIR = DATASETS_DIR / "tiled_images"
REMAPPED_GT_DIR = DATASETS_DIR / "remapped_landcover_masks"
PRED_DIR = DATASETS_DIR / f"pred_masks_{model_safe}"
METRICS_CSV = DATASETS_DIR / f"landcover_metrics_scores_{model_safe}.csv"
TRIPLET_DIR = DATASETS_DIR / f"triplet_comparisons_{model_safe}"
CKPT_PATH = PROJECT_ROOT / "checkpoints" / f"pipeline_best_{model_safe}_best.pt"


def clean_mask_tifs(dir_path: Path) -> None:
    dir_path.mkdir(parents=True, exist_ok=True)
    for pat in ("*.tif", "*.tiff"):
        for p in dir_path.glob(pat):
            p.unlink(missing_ok=True)


def clean_pngs(dir_path: Path) -> None:
    dir_path.mkdir(parents=True, exist_ok=True)
    for p in dir_path.glob("*.png"):
        p.unlink(missing_ok=True)


clean_mask_tifs(TILED_RGB_DIR)
clean_mask_tifs(REMAPPED_GT_DIR)

tile_script = PROJECT_ROOT / "landcover_verification" / "tile_landcover_for_verification.py"
subprocess.run(
    [
        sys.executable,
        str(tile_script),
        "--images-dir",
        str(RAW_RGB_DIR),
        "--masks-dir",
        str(RAW_GT_DIR),
        "--out-images-dir",
        str(TILED_RGB_DIR),
        "--out-remapped-masks-dir",
        str(REMAPPED_GT_DIR),
        "--tile-size",
        str(TILE_SIZE),
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)
print("Tiling and remap complete.")

Paired source scenes: 41
Observed raw labels: [0, 1, 2, 3, 4]
Total generated tiles: 12737
Wrote RGB tiles: 12737 -> /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/tiled_images
Wrote remapped mask tiles: 12737 -> /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/remapped_landcover_masks
Tiling and remap complete.


step 3: run inference on tiled RGB with runmodel. Requires the previous code cell (path variables and tiling). Writes predictions under `./landcover_verification/datasets/pred_masks_unet` for the default model. Re-running clears that prediction directory of old GeoTIFFs first.

In [16]:
import subprocess
import sys

assert CKPT_PATH.is_file(), f"Missing checkpoint: {CKPT_PATH}"
assert TILED_RGB_DIR.is_dir(), f"Missing tiled RGB: {TILED_RGB_DIR}"
assert REMAPPED_GT_DIR.is_dir(), f"Missing remapped GT: {REMAPPED_GT_DIR}"

clean_mask_tifs(PRED_DIR)
runmodel_main = PROJECT_ROOT / "runmodel" / "main.py"
subprocess.run(
    [
        sys.executable,
        str(runmodel_main),
        "--model",
        MODEL,
        "--ckpt",
        str(CKPT_PATH),
        "--images_dir",
        str(TILED_RGB_DIR),
        "--masks_dir",
        str(REMAPPED_GT_DIR),
        "--pred_output_dir",
        str(PRED_DIR),
        "--limit",
        str(RUNMODEL_LIMIT),
        "--inference-patch-size",
        str(INFERENCE_PATCH_SIZE),
        "--inference-overlap",
        str(INFERENCE_OVERLAP),
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)
print("Runmodel complete.")

Loading checkpoint: /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/checkpoints/pipeline_best_unet_best.pt
Saved preds to: /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/pred_masks_unet
Runmodel complete.


step 4: score per-tile F1 and IoU between predictions and remapped GT; writes `./landcover_verification/datasets/landcover_metrics_scores_unet.csv` for the default model.

In [17]:
import subprocess
import sys

assert PRED_DIR.is_dir(), f"Missing predictions: {PRED_DIR}"
assert REMAPPED_GT_DIR.is_dir(), f"Missing remapped GT: {REMAPPED_GT_DIR}"

score_script = PROJECT_ROOT / "landcover_verification" / "score_landcover_metrics.py"
subprocess.run(
    [
        sys.executable,
        str(score_script),
        "--pred-dir",
        str(PRED_DIR),
        "--gt-dir",
        str(REMAPPED_GT_DIR),
        "--out-csv",
        str(METRICS_CSV),
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)
print(f"Scores written to {METRICS_CSV}")

Scored tiles: 12737
Skipped tiles: 0
Macro F1: 0.286414
F1 class 0: 0.482531
F1 class 1: 0.041119
F1 class 2: 0.372608
F1 class 3: 0.249399
Macro IoU: 0.266658
IoU class 0: 0.473803
IoU class 1: 0.030458
IoU class 2: 0.314820
IoU class 3: 0.247551
CSV saved: /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/landcover_metrics_scores_unet.csv
Scores written to /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/landcover_metrics_scores_unet.csv


step 5: build triplet PNG montages (RGB | colored GT | colored pred), `--limit 3`, `--id-source pred` (strict; errors if a counterpart is missing). Output under `./landcover_verification/datasets/triplet_comparisons_unet/` for the default model. Re-running clears old PNGs in that folder first.

In [18]:
import subprocess
import sys

assert TILED_RGB_DIR.is_dir(), f"Missing tiled RGB: {TILED_RGB_DIR}"
assert REMAPPED_GT_DIR.is_dir(), f"Missing remapped GT: {REMAPPED_GT_DIR}"
assert PRED_DIR.is_dir(), f"Missing predictions: {PRED_DIR}"

clean_pngs(TRIPLET_DIR)
trip_script = PROJECT_ROOT / "landcover_verification" / "compare_landcover_triplets.py"
subprocess.run(
    [
        sys.executable,
        str(trip_script),
        "--images-dir",
        str(TILED_RGB_DIR),
        "--gt-dir",
        str(REMAPPED_GT_DIR),
        "--pred-dir",
        str(PRED_DIR),
        "--output-dir",
        str(TRIPLET_DIR),
        "--limit",
        str(TRIPLET_LIMIT),
        "--id-source",
        "pred",
    ],
    check=True,
    cwd=str(PROJECT_ROOT),
)
print(f"Triplets written to {TRIPLET_DIR}")

Wrote 3 triplet comparison PNGs to /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/triplet_comparisons_unet
Panel order: RGB | GT (colored remapped mask) | PRED (colored prediction)
Triplets written to /home/ehurd1@cfreg.local/ndvi-guided-rgb-segmentation/landcover_verification/datasets/triplet_comparisons_unet
